# VIIRS vegetation hotspots → ArcGIS Online

A self-contained hourly workflow based on the production R code in
[kengelmayer/VIIRS_Filter](https://github.com/kengelmayer/VIIRS_Filter/tree/2993200916a64dcb0cd456d1fae5ad78bd4b7c97)
(`R/update_viirs.R`, `R/lib_viirs.R`, and the production workflow settings).

**VIIRS → unseen detections → 3×3 WorldCover filter → your hosted country layer → hosted point layer.**

Keeps nominal/high-confidence detections with at least 50% vegetation among valid
footprint samples, rejects built-up centers, and retains a rolling seven days.
It creates one hosted layer on the first run, then updates that same layer.

Use a current **ArcGIS Online Standard notebook runtime** with ArcGIS API for Python,
NumPy, Rasterio, Requests and Shapely 2 or later. No ArcPy or Spatial Analyst is used.
The notebook signs in with `GIS("home")`; passwords and API keys are unnecessary.
The owner needs notebook execution/scheduling and hosted-layer publishing privileges.

**Before running:** replace the country item placeholder below. Use **Run All** for
each execution. After the first successful run, copy the printed target item ID into
`TARGET_ITEM_ID` and save. Hourly scheduling instructions are at the end.

## 1. Settings

`COUNTRIES_ITEM_ID` is the 32-character item ID from your layer's item page.
`COUNTRIES_LAYER_ID` is the numeric sublayer ID in the REST URL (often `0`).
`COUNTRY_NAME_FIELD` defaults to the original Natural Earth field, `SOVEREIGNT`.
Use the actual field name, not its display alias. Countries label accepted points;
points outside the polygons remain with an empty country, matching R.

The production R workflow starts with the newest source age block plus one hour.
This remains the default (`BOOTSTRAP_HOURS = 0`). For the initial run, set it to `168`
if you want all history currently available from the VIIRS source. Source retention
can be shorter than seven days; an empty installation cannot recover unavailable
history. Daily reconciliation subsequently checks all available records in the window.

Change `SERVICE_NAME` and leave `TARGET_ITEM_ID` blank to create an independent
variant with different filter/country settings. Existing detections keep their original
classification, as in R. Tuning, retention and bootstrap settings can be changed in place.

In [ ]:
# REQUIRED: hosted Feature Layer item ID (32 characters), not a web map ID.
COUNTRIES_ITEM_ID = "PASTE_YOUR_COUNTRY_FEATURE_LAYER_ITEM_ID_HERE"
COUNTRIES_LAYER_ID = 0              # Numeric sublayer ID in its REST URL.
COUNTRY_NAME_FIELD = "SOVEREIGNT"   # Change if your hosted layer uses another name.

# Leave blank on first run. The notebook creates and remembers its own output.
# After creation, copy the printed item ID here for explicit, durable binding.
TARGET_ITEM_ID = ""
SERVICE_NAME = "viirs_vegetation_hourly_v2"
SERVICE_TITLE = "VIIRS vegetation hotspots — last 7 days"

ROLLING_HOURS = 168
SOURCE_OVERLAP_HOURS = 1
BOOTSTRAP_HOURS = 0                 # 0 = latest source block, as in production R.
# Set BOOTSTRAP_HOURS = 168 before the FIRST run to fetch available source history.
RECONCILE_EVERY_HOURS = 24           # Full available rolling-window query once daily.
CONFIDENCE_VALUES = ("nominal", "high")
LANDCOVER_GRID_SIZE = 3             # 3 = original 3×3 footprint; 1 = center only.
VEGETATION_SHARE_THRESHOLD = 0.5
REJECT_BUILT_CENTER = True
VEGETATION_CODES = (10, 20, 30, 40, 90, 95, 100)
UPDATE_HOURS_OLD = True             # One server-side calculate; preserves R output.

SOURCE_WORKERS = 4
RASTER_WORKERS = 4
SOURCE_PAGE_SIZE = 4000
PROCESS_BATCH_SIZE = 10000
APPEND_BATCH_SIZE = 5000
COUNTRY_CACHE_HOURS = 24
# Use workspace MUST be enabled in the scheduled task. One task per output layer.
WORKSPACE_ROOT = "/arcgis/home/viirs_filter"

VIIRS_URL = (
    "https://services9.arcgis.com/RHVPKKiFTONKtxq3/arcgis/rest/services/"
    "Satellite_VIIRS_Thermal_Hotspots_and_Fire_Activity/FeatureServer/0"
)
WORLDCOVER_TEMPLATE = (
    "https://esa-worldcover.s3.eu-central-1.amazonaws.com/v200/2021/map/"
    "ESA_WorldCover_10m_2021_v200_{tile}_Map.tif"
)


## 2. Runtime, validation and persistent state

The following cells define the workflow; the final run cell performs the work.
Accepted **and rejected** detection keys are cached in SQLite under `/arcgis/home`.
Successful batches persist independently, so retries avoid repeating completed work.
A workspace lock prevents overlapping runs that use the same state directory.

Keep one scheduled task and one notebook owner for each output service. Separate
notebooks/workspaces targeting the same service are not coordinated by this lock.
If workspace state is lost, the notebook recovers accepted keys from the hosted layer;
rejected points must be sampled again. The target item ID should be saved in settings.
Use the output exclusively for this workflow; if its rows are manually removed, clear
its state directory while the task is paused so those rows can be recovered if available.

In [ ]:
import hashlib
import json
import math
import re
import sqlite3
import threading
import time
from collections import defaultdict
from concurrent.futures import ThreadPoolExecutor, as_completed
from contextlib import contextmanager
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import rasterio
import requests
import shapely
from rasterio.windows import Window
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
from shapely.geometry import shape
from shapely.strtree import STRtree

if int(shapely.__version__.split(".")[0]) < 2:
    raise RuntimeError("Choose a current Standard runtime with Shapely >= 2.")

HOUR_MS = 3_600_000
ALGORITHM_VERSION = "r-footprint-v1"
CLASS_NAMES = {
    10: "Tree cover", 20: "Shrubland", 30: "Grassland", 40: "Cropland",
    50: "Built-up", 60: "Bare or sparse vegetation", 70: "Snow and ice",
    80: "Permanent water bodies", 90: "Herbaceous wetland",
    95: "Mangroves", 100: "Moss and lichen",
}


def log(message):
    print(f"{datetime.now(timezone.utc):%H:%M:%S} UTC | {message}", flush=True)


def chunks(values, size):
    for start in range(0, len(values), size):
        yield values[start:start + size]


def sql_time(ms):
    return "TIMESTAMP '" + datetime.fromtimestamp(ms / 1000, timezone.utc).strftime(
        "%Y-%m-%d %H:%M:%S") + "'"


def sql_string(value):
    return "'" + str(value).replace("'", "''") + "'"


def require_success(response, action):
    if response is True or (isinstance(response, dict) and response.get("success") is True):
        return
    raise RuntimeError(f"{action} failed: {response}")


def config_signature():
    # These change classification. Retention/bootstrap/tuning can change independently.
    config = [ALGORITHM_VERSION, VIIRS_URL, WORLDCOVER_TEMPLATE,
              COUNTRIES_ITEM_ID, COUNTRIES_LAYER_ID, COUNTRY_NAME_FIELD,
              sorted(CONFIDENCE_VALUES), LANDCOVER_GRID_SIZE,
              VEGETATION_SHARE_THRESHOLD, REJECT_BUILT_CENTER, sorted(VEGETATION_CODES)]
    return hashlib.sha256(json.dumps(config, sort_keys=True).encode()).hexdigest()[:24]


def validate_config():
    if not re.fullmatch(r"[a-fA-F0-9]{32}", COUNTRIES_ITEM_ID):
        raise ValueError("Replace COUNTRIES_ITEM_ID in the settings cell with your hosted country layer item ID.")
    if TARGET_ITEM_ID and not re.fullmatch(r"[a-fA-F0-9]{32}", TARGET_ITEM_ID):
        raise ValueError("TARGET_ITEM_ID must be blank or a 32-character item ID.")
    if not re.fullmatch(r"[A-Za-z][A-Za-z0-9_]{2,90}", SERVICE_NAME):
        raise ValueError("SERVICE_NAME must use letters, digits and underscores (3–91 characters).")
    if not re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*", COUNTRY_NAME_FIELD):
        raise ValueError("Use the actual country field name, not its alias.")
    if not isinstance(COUNTRIES_LAYER_ID, int) or COUNTRIES_LAYER_ID < 0:
        raise ValueError("COUNTRIES_LAYER_ID must be a nonnegative integer.")
    if LANDCOVER_GRID_SIZE not in (1, 3):
        raise ValueError("LANDCOVER_GRID_SIZE must be 1 or 3.")
    if not (1 <= ROLLING_HOURS <= 168 and 0 <= BOOTSTRAP_HOURS <= 168):
        raise ValueError("Rolling/bootstrap windows must be within 168 hours.")
    if not (0 <= SOURCE_OVERLAP_HOURS <= 12 and 0 <= VEGETATION_SHARE_THRESHOLD <= 1):
        raise ValueError("Invalid overlap or vegetation share threshold.")
    if not CONFIDENCE_VALUES or not set(CONFIDENCE_VALUES) <= {"low", "nominal", "high"}:
        raise ValueError("Confidence classes must be low, nominal, or high.")
    if not VEGETATION_CODES or not set(VEGETATION_CODES) <= CLASS_NAMES.keys():
        raise ValueError("VEGETATION_CODES contains an unknown WorldCover class.")
    for value in [SOURCE_WORKERS, RASTER_WORKERS, SOURCE_PAGE_SIZE,
                  PROCESS_BATCH_SIZE, APPEND_BATCH_SIZE, COUNTRY_CACHE_HOURS,
                  RECONCILE_EVERY_HOURS]:
        if not isinstance(value, int) or value < 1:
            raise ValueError("Worker, batch, and cache settings must be positive integers.")
    if SOURCE_WORKERS > 8 or RASTER_WORKERS > 8:
        raise ValueError("Use at most 8 source/raster workers to bound memory and connections.")
    if APPEND_BATCH_SIZE > 10000:
        raise ValueError("Keep APPEND_BATCH_SIZE <= 10000 to bound request size.")


_http_local = threading.local()


def public_session():
    if not hasattr(_http_local, "session"):
        session = requests.Session()
        # Only read operations use this session. POST here is an ArcGIS query.
        retries = Retry(total=4, backoff_factor=1, status_forcelist=[429, 500, 502, 503, 504],
                        allowed_methods=["GET", "POST", "HEAD"], respect_retry_after_header=True)
        session.mount("https://", HTTPAdapter(max_retries=retries))
        session.headers["User-Agent"] = "VIIRS-AGOL-notebook/2.0"
        _http_local.session = session
    return _http_local.session


def public_json(url, params=None):
    response = public_session().get(url, params={"f": "json", **(params or {})}, timeout=(20, 180))
    response.raise_for_status()
    data = response.json()
    if "error" in data:
        raise RuntimeError(f"Public ArcGIS query failed: {data['error']}")
    return data


def atomic_json(path, data):
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(json.dumps(data, ensure_ascii=False, allow_nan=False), encoding="utf-8")
    temporary.replace(path)


@contextmanager
def state_db(directory):
    # SQLite locking coordinates containers sharing this workspace. No stale PID locks.
    # Deliberately one writer/task for an output service; not a distributed AGOL lock.
    directory.mkdir(parents=True, exist_ok=True)
    lock = sqlite3.connect(directory / "run_lock.sqlite", timeout=0)
    db = None
    try:
        lock.execute("CREATE TABLE IF NOT EXISTS lock_row (id INTEGER)")
        lock.execute("BEGIN EXCLUSIVE")
        db = sqlite3.connect(directory / "state.sqlite")
        db.executescript("""
            CREATE TABLE IF NOT EXISTS meta (key TEXT PRIMARY KEY, value TEXT NOT NULL);
            CREATE TABLE IF NOT EXISTS seen (
                detection_id TEXT PRIMARY KEY, acq_time INTEGER NOT NULL, accepted INTEGER NOT NULL);
            CREATE INDEX IF NOT EXISTS seen_time ON seen(acq_time);
        """)
        yield db
    finally:
        if db is not None:
            db.close()
        lock.close()


def meta_get(db, key, default=None):
    row = db.execute("SELECT value FROM meta WHERE key=?", (key,)).fetchone()
    return json.loads(row[0]) if row else default


def meta_set(db, key, value):
    db.execute("INSERT OR REPLACE INTO meta VALUES (?, ?)", (key, json.dumps(value)))


def remember(db, rows, accepted_ids):
    db.executemany("INSERT OR REPLACE INTO seen VALUES (?, ?, ?)",
                   [(r["detection_id"], r["acq_time"], int(r["detection_id"] in accepted_ids)) for r in rows])
    db.commit()


def unknown_rows(db, rows):
    known = set()
    for batch in chunks([r["detection_id"] for r in rows], 800):
        placeholders = ",".join("?" for _ in batch)
        known.update(r[0] for r in db.execute(
            f"SELECT detection_id FROM seen WHERE detection_id IN ({placeholders})", batch))
    return [r for r in rows if r["detection_id"] not in known]


## 3. Incremental VIIRS retrieval

The source query filters confidence and acquisition time on the server. It snapshots
OBJECTIDs, fetches bounded pages concurrently, and verifies that the source did not
refresh mid-download. Missing pages fail the run rather than advancing the checkpoint.

The normal query uses the newest source age, overlap, and elapsed time since the
last completed run. A daily full-window query also catches older late arrivals still
available at the source. Cached keys avoid repeating their spatial processing.

In [ ]:
def select_source_age(min_age, now_ms, last_success_ms, last_reconcile_ms, last_window):
    full = (last_success_ms is not None and (
        last_reconcile_ms is None or
        now_ms - last_reconcile_ms >= RECONCILE_EVERY_HOURS * HOUR_MS or
        ROLLING_HOURS > last_window))
    if full:
        return ROLLING_HOURS + 1, True
    if last_success_ms is None:
        age = max(math.ceil(min_age) + SOURCE_OVERLAP_HOURS, BOOTSTRAP_HOURS)
    else:
        gap = max(0, math.ceil((now_ms - last_success_ms) / HOUR_MS))
        age = math.ceil(min_age) + SOURCE_OVERLAP_HOURS + gap
    return min(age, ROLLING_HOURS + 1), False


def fetch_source(db, now_ms):
    required = {"acq_time", "satellite", "confidence", "scan", "track", "frp", "hours_old"}
    # Snapshot OBJECTIDs, then query bounded ID batches: no offset drift during refresh.
    # If the publisher replaces records while reading, retry the complete snapshot.
    for attempt in range(3):
        metadata = public_json(VIIRS_URL)
        fields = {f["name"] for f in metadata.get("fields", [])}
        if not required <= fields:
            raise RuntimeError(f"VIIRS source schema changed: missing {required - fields}")
        oid = metadata["objectIdField"]
        stats = public_json(VIIRS_URL + "/query", {
            "where": "1=1", "returnGeometry": "false",
            "outStatistics": json.dumps([{"statisticType": "min", "onStatisticField": "hours_old",
                                          "outStatisticFieldName": "min_age"}])})
        values = stats.get("features", [])
        min_age = values[0]["attributes"].get("min_age") if values else None
        if min_age is None:
            raise RuntimeError("VIIRS source is empty/unavailable; checkpoint and output are unchanged.")
        max_age, full = select_source_age(
            min_age, now_ms, meta_get(db, "last_success_ms"),
            meta_get(db, "last_reconcile_ms"), meta_get(db, "rolling_hours", ROLLING_HOURS))
        where = (f"hours_old <= {max_age} AND confidence IN "
                 f"({','.join(sql_string(v) for v in CONFIDENCE_VALUES)}) "
                 f"AND acq_time >= {sql_time(now_ms - ROLLING_HOURS * HOUR_MS)} "
                 f"AND acq_time <= {sql_time(now_ms + HOUR_MS)}")
        id_response = public_json(VIIRS_URL + "/query", {"where": where, "returnIdsOnly": "true"})
        if "objectIds" not in id_response or id_response.get("exceededTransferLimit"):
            raise RuntimeError("Source did not return a complete ID snapshot.")
        ids = sorted(set(id_response["objectIds"] or []))
        if len(ids) >= 1_000_000:
            raise RuntimeError("Source ID snapshot reached the service limit; narrow the source window.")
        page_size = min(SOURCE_PAGE_SIZE, int(metadata.get("maxRecordCount", 2000)))

        def fetch_batch(batch):
            response = public_session().post(VIIRS_URL + "/query", data={
                "f": "json", "objectIds": ",".join(map(str, batch)), "where": "1=1",
                "outFields": ",".join([oid] + sorted(required)), "returnGeometry": "true",
                "outSR": "4326"}, timeout=(20, 180))
            response.raise_for_status()
            payload = response.json()
            if "error" in payload or payload.get("exceededTransferLimit"):
                raise RuntimeError(f"Source page failed or was truncated: {payload.get('error', 'limit')}")
            return payload.get("features", [])

        features = []
        with ThreadPoolExecutor(max_workers=SOURCE_WORKERS) as pool:
            for page in pool.map(fetch_batch, chunks(ids, page_size)):
                features.extend(page)
        after = public_json(VIIRS_URL)
        stamp = lambda p: p.get("editingInfo", {}).get("lastEditDate")
        returned_ids = [f["attributes"][oid] for f in features]
        if (set(returned_ids) != set(ids) or len(returned_ids) != len(ids) or
                (stamp(metadata) is not None and stamp(metadata) != stamp(after))):
            log(f"VIIRS refreshed during read; retrying snapshot ({attempt + 1}/3).")
            continue
        rows = {}
        for feature in sorted(features, key=lambda f: f["attributes"][oid]):
            a, g = feature["attributes"], feature.get("geometry") or {}
            try:
                lon, lat, acq = float(g["x"]), float(g["y"]), int(a["acq_time"])
                if not (math.isfinite(lon) and math.isfinite(lat) and -180 <= lon <= 180 and -90 <= lat <= 90):
                    raise ValueError("invalid coordinates")
            except (ValueError, TypeError, KeyError) as exc:
                raise RuntimeError("VIIRS returned an invalid geometry or acquisition time.") from exc
            if not now_ms - ROLLING_HOURS * HOUR_MS <= acq <= now_ms + HOUR_MS:
                continue
            satellite = str(a.get("satellite") or "")
            key = f"{satellite}|{acq}|{lon:.5f}|{lat:.5f}"
            def number(value):
                return float(value) if value is not None and math.isfinite(float(value)) else None
            rows.setdefault(key, {"detection_id": key, "longitude": lon, "latitude": lat,
                                  "acq_time": acq, "frp": number(a.get("frp")),
                                  "scan": number(a.get("scan")), "track": number(a.get("track"))})
        log(f"Source age <= {max_age} h; {len(rows):,} unique detections; full reconciliation={full}.")
        return list(rows.values()), full, {"min_age": min_age, "max_age": max_age}
    raise RuntimeError("VIIRS changed during all three reads; rerun after its refresh completes.")


## 4. WorldCover footprint filter

This ports the current R method: use scan/track in kilometres (375 m if missing),
latitude-adjusted east/west distances, and offsets of −⅓, 0, +⅓ of each footprint
dimension. Longitude wraps at the dateline. The 3×3 grid is an approximation, not
the oriented satellite pixel or an exact area fraction.

WorldCover 2021 v200 COGs are read at native resolution using HTTP range requests.
Samples are grouped by tile and TIFF block so a required block is read once within
a batch. Four tile workers overlap network I/O. No raster analysis service is called.

No-data samples are excluded from the denominator. A missing center can pass when
the other valid samples qualify; a built-up center cannot. All-no-data points fail.
Only a confirmed HTTP 404 is treated as an absent tile; other read errors stop the run.

In [ ]:
def footprint_samples(rows):
    fractions = [0.0] if LANDCOVER_GRID_SIZE == 1 else [-1/3, 0.0, 1/3]
    offsets = np.array([(x, y) for y in fractions for x in fractions])
    n = len(offsets)
    lon = np.repeat([r["longitude"] for r in rows], n)
    lat = np.repeat([r["latitude"] for r in rows], n)
    scan = np.repeat([375.0 if r["scan"] is None else r["scan"] * 1000 for r in rows], n)
    track = np.repeat([375.0 if r["track"] is None else r["track"] * 1000 for r in rows], n)
    dx, dy = np.tile(offsets, (len(rows), 1)).T
    x = ((lon + dx * scan / (111320 * np.maximum(np.abs(np.cos(np.deg2rad(lat))), 0.01)) + 180) % 360) - 180
    y = lat + dy * track / 111320
    return x, y, n


def tile_name(lon, lat):
    x = int(math.floor(min(max(lon, -180), 180 - 1e-10) / 3) * 3)
    y = int(math.floor(min(max(lat, -90), 90 - 1e-10) / 3) * 3)
    return f"{'S' if y < 0 else 'N'}{abs(y):02d}{'W' if x < 0 else 'E'}{abs(x):03d}"


def read_blocks(dataset, x, y):
    """Native-resolution samples; read each required TIFF block once per tile/batch."""
    result = np.full(len(x), -1, dtype=np.int16)
    rr, cc = rasterio.transform.rowcol(dataset.transform, x, y)
    rr, cc = np.asarray(rr), np.asarray(cc)
    valid = np.flatnonzero((rr >= 0) & (rr < dataset.height) & (cc >= 0) & (cc < dataset.width))
    block_h, block_w = dataset.block_shapes[0]
    groups = defaultdict(list)
    for i in valid:
        groups[(int(rr[i] // block_h), int(cc[i] // block_w))].append(i)
    for (br, bc), positions in groups.items():
        positions = np.asarray(positions)
        r0, c0 = br * block_h, bc * block_w
        block = dataset.read(1, window=Window(c0, r0, min(block_w, dataset.width-c0),
                                             min(block_h, dataset.height-r0)), masked=True)
        samples = block[rr[positions]-r0, cc[positions]-c0]
        usable = ~np.ma.getmaskarray(samples) & (np.ma.getdata(samples) != 0)
        result[positions[usable]] = np.ma.getdata(samples)[usable].astype(np.int16)
    return result


def sample_one_tile(tile, x, y):
    url = WORLDCOVER_TEMPLATE.format(tile=tile)
    options = dict(GDAL_DISABLE_READDIR_ON_OPEN="EMPTY_DIR", CPL_VSIL_CURL_ALLOWED_EXTENSIONS=".tif",
                   GDAL_HTTP_MULTIRANGE="YES", GDAL_HTTP_MERGE_CONSECUTIVE_RANGES="YES",
                   GDAL_HTTP_MAX_RETRY="3", GDAL_HTTP_RETRY_DELAY="1",
                   GDAL_HTTP_CONNECTTIMEOUT="20", GDAL_HTTP_TIMEOUT="120",
                   CPL_VSIL_CURL_CACHE_SIZE=64*1024*1024, GDAL_CACHEMAX=128*1024*1024)
    with rasterio.Env(**options):
        try:
            with rasterio.open(url) as dataset:
                if dataset.crs != rasterio.crs.CRS.from_epsg(4326):
                    raise RuntimeError(f"Unexpected WorldCover CRS for {tile}.")
                return read_blocks(dataset, x, y)
        except rasterio.errors.RasterioIOError:
            # Only a confirmed absent tile is no-data. Timeouts/403/5xx fail the run.
            probe = public_session().head(url, timeout=(20, 60), allow_redirects=True)
            if probe.status_code == 404:
                return np.full(len(x), -1, dtype=np.int16)
            probe.raise_for_status()
            raise


def classify_samples(rows, codes):
    valid = codes >= 0
    count = valid.sum(axis=1)
    vegetation = np.isin(codes, VEGETATION_CODES).sum(axis=1)
    shares = np.divide(vegetation, count, out=np.zeros(len(rows), dtype=float), where=count > 0)
    center = codes[:, codes.shape[1] // 2]
    keep = (count > 0) & (shares >= VEGETATION_SHARE_THRESHOLD)
    if REJECT_BUILT_CENTER:
        keep &= center != 50
    accepted = []
    for i in np.flatnonzero(keep):
        accepted.append({**rows[i], "landcover_center_class": CLASS_NAMES.get(int(center[i]), "Unknown")})
    return accepted


def vegetation_filter(rows):
    if not rows:
        return []
    x, y, n = footprint_samples(rows)
    values = np.full(len(x), -1, dtype=np.int16)
    groups = defaultdict(list)
    for i in np.flatnonzero((y >= -90) & (y <= 90)):
        groups[tile_name(x[i], y[i])].append(i)
    with ThreadPoolExecutor(max_workers=RASTER_WORKERS) as pool:
        futures = {pool.submit(sample_one_tile, tile, x[ix], y[ix]): ix for tile, ix in groups.items()}
        done = 0
        for future in as_completed(futures):
            indices = futures[future]
            values[indices] = future.result()  # Propagate any download/read failure.
            done += 1
            if done % 25 == 0 or done == len(groups):
                log(f"WorldCover tiles: {done}/{len(groups)}")
    accepted = classify_samples(rows, values.reshape(len(rows), n))
    log(f"WorldCover accepted {len(accepted):,}/{len(rows):,} new detections.")
    return accepted


## 5. Countries from your content

The country layer is queried using the notebook owner's login. Geometry is requested
as WGS84 GeoJSON so multipart polygons and holes are preserved. A cached spatial index
labels only accepted detections. The cache is refreshed after 24 hours or a reported
layer edit. At shared borders or overlapping polygons, the smallest country OBJECTID
wins deterministically; this may differ from R's implicit feature ordering.

In [ ]:
def country_layer(gis):
    item = gis.content.get(COUNTRIES_ITEM_ID)
    if item is None:
        raise ValueError("Country item is missing or inaccessible to the notebook owner.")
    layers = [layer for layer in item.layers if int(layer.properties.id) == COUNTRIES_LAYER_ID]
    if len(layers) != 1:
        raise ValueError(f"Country sublayer {COUNTRIES_LAYER_ID} was not found.")
    layer = layers[0]
    if layer.properties.geometryType != "esriGeometryPolygon":
        raise ValueError("Country layer must contain polygons.")
    fields = {f["name"]: f for f in layer.properties.fields}
    if COUNTRY_NAME_FIELD not in fields:
        raise ValueError(f"Country field {COUNTRY_NAME_FIELD!r} missing. Fields: {list(fields)}")
    if fields[COUNTRY_NAME_FIELD]["type"] != "esriFieldTypeString":
        raise ValueError("COUNTRY_NAME_FIELD must be a text field.")
    return layer


def country_index(gis, layer, directory, now_ms):
    path = directory / "countries_cache.json"
    edit_stamp = layer.properties.get("editingInfo", {}).get("lastEditDate")
    identity = [COUNTRIES_ITEM_ID, COUNTRIES_LAYER_ID, COUNTRY_NAME_FIELD, edit_stamp]
    cached = None
    if path.exists():
        try:
            candidate = json.loads(path.read_text(encoding="utf-8"))
            if candidate["identity"] == identity and now_ms - candidate["loaded_ms"] < COUNTRY_CACHE_HOURS * HOUR_MS:
                cached = candidate
        except (ValueError, KeyError):
            pass  # Re-download a corrupt/obsolete cache; do not use partial boundaries.
    if cached is None:
        ids_result = layer.query(where="1=1", return_ids_only=True)
        ids = sorted(ids_result.get("objectIds") or [])
        if not ids or ids_result.get("exceededTransferLimit") or len(ids) >= 1_000_000:
            raise RuntimeError("Country layer is empty or its ID query is incomplete.")
        oid = layer.properties.objectIdField
        features = []
        page_size = min(100, int(layer.properties.get("maxRecordCount", 1000)))
        for batch in chunks(ids, page_size):
            # Ask the service to project and convert its rings/holes to GeoJSON.
            response = gis.session.post(layer.url + "/query", data={
                "f": "geojson", "objectIds": ",".join(map(str, batch)),
                "outFields": f"{oid},{COUNTRY_NAME_FIELD}", "outSR": "4326",
                "returnGeometry": "true"}, timeout=(20, 180))
            response.raise_for_status()
            payload = response.json()
            page = payload.get("features", [])
            if (payload.get("type") != "FeatureCollection" or payload.get("exceededTransferLimit") or
                    len(page) != len(batch)):
                raise RuntimeError(f"Incomplete country download: {payload.get('error', 'truncated page')}")
            features.extend(page)
        returned = [f["properties"][oid] for f in features]
        if set(returned) != set(ids) or len(returned) != len(ids):
            raise RuntimeError("Country IDs changed during download; rerun after its update.")
        # Store/query order follows OBJECTID: deterministic first match at shared borders.
        features.sort(key=lambda f: f["properties"][oid])
        cached = {"identity": identity, "loaded_ms": now_ms, "features": features}
        atomic_json(path, cached)
        log(f"Cached {len(features)} country polygons from your hosted layer.")
    geometries, names = [], []
    for feature in cached["features"]:
        if feature.get("geometry") is None:
            raise RuntimeError("Country layer contains a null geometry; repair it before running.")
        geometry = shape(feature["geometry"])
        if geometry.geom_type not in ("Polygon", "MultiPolygon") or geometry.is_empty:
            raise RuntimeError("Country layer contains an empty/nonpolygon geometry.")
        if not geometry.is_valid:
            geometry = shapely.make_valid(geometry)
        geometries.append(geometry)
        name = str(feature["properties"].get(COUNTRY_NAME_FIELD) or "")
        if len(name) > 255:
            raise ValueError("A country name exceeds the output field's 255-character limit.")
        names.append(name)
    return STRtree(geometries), np.array(names, dtype=object)


def assign_countries(rows, index):
    if not rows:
        return rows
    tree, names = index
    points = shapely.points([r["longitude"] for r in rows], [r["latitude"] for r in rows])
    pairs = tree.query(points, predicate="intersects")
    chosen = np.full(len(rows), len(names), dtype=np.int64)
    if pairs.size:
        np.minimum.at(chosen, pairs[0], pairs[1])
    for i, row in enumerate(rows):
        row["country"] = str(names[chosen[i]]) if chosen[i] < len(names) else ""
    return rows


## 6. Create and update the hosted point layer

The layer has the original dashboard attributes plus `detection_id`, which has a
unique index. `acq_time` is a proper ArcGIS UTC date rather than ISO text.
An acquisition-time index helps retention and time queries. The service is created
privately and its item ID, URL, feature OBJECTIDs, and map references remain stable.

Batched append/upsert inserts new keys and skips existing keys. Expired detections
are deleted after processing. `hours_old` is recalculated in one server operation.
For maximum write efficiency, set `UPDATE_HOURS_OLD = False` and use relative date
filters on `acq_time`; then ignore the stored age field. Existing numeric age values
will stop changing, and new age values are null.

A failed batch or maintenance operation raises an error and leaves the success
checkpoint unchanged. The complete hourly run is **not one atomic transaction**:
previous successful batches can already be visible. Retry with **Run All** to finish.
The layer is never truncated as part of an update.

In [ ]:
FIELDS = [
    {"name": "OBJECTID", "type": "esriFieldTypeOID", "alias": "OBJECTID", "nullable": False, "editable": False},
    {"name": "detection_id", "type": "esriFieldTypeString", "alias": "Detection ID", "length": 160, "nullable": False},
    {"name": "acq_time", "type": "esriFieldTypeDate", "alias": "Acquisition time (UTC)", "nullable": False},
    {"name": "hours_old", "type": "esriFieldTypeDouble", "alias": "Age at last update (hours)"},
    {"name": "frp", "type": "esriFieldTypeDouble", "alias": "Fire radiative power (MW)"},
    {"name": "landcover_center_class", "type": "esriFieldTypeString", "alias": "WorldCover center class", "length": 64},
    {"name": "country", "type": "esriFieldTypeString", "alias": "Country", "length": 255},
]
DATA_FIELDS = [field for field in FIELDS if field["name"] != "OBJECTID"]


def layer_definition():
    return {
        "id": 0, "name": "VIIRS vegetation hotspots", "type": "Feature Layer",
        "geometryType": "esriGeometryPoint", "objectIdField": "OBJECTID",
        "displayField": "country", "fields": FIELDS,
        "hasZ": False, "hasM": False,
        "extent": {"xmin": -180, "ymin": -90, "xmax": 180, "ymax": 90,
                   "spatialReference": {"wkid": 4326}},
        "spatialReference": {"wkid": 4326},
        "drawingInfo": {"renderer": {"type": "simple", "symbol": {
            "type": "esriSMS", "style": "esriSMSCircle", "color": [230, 80, 20, 190],
            "size": 4, "outline": {"color": [255, 255, 255, 100], "width": 0.3}}}},
        "timeInfo": {"startTimeField": "acq_time", "endTimeField": None,
                     "trackIdField": "", "timeReference": {"timeZone": "UTC", "respectsDaylightSaving": False}},
        "indexes": [{"name": "ux_detection_id", "fields": "detection_id", "isAscending": True, "isUnique": True},
                    {"name": "ix_acq_time", "fields": "acq_time", "isAscending": True, "isUnique": False}],
    }


def get_target(gis, db):
    from arcgis.features import FeatureLayerCollection, FeatureLayer
    owner = gis.users.me.username
    service_tag = f"viirs-pipeline-{SERVICE_NAME}"
    config_tag = f"viirs-config-{config_signature()}"
    saved_id = meta_get(db, "target_item_id")
    if saved_id and TARGET_ITEM_ID and saved_id != TARGET_ITEM_ID:
        raise ValueError("TARGET_ITEM_ID disagrees with saved state. Use a new SERVICE_NAME for a different output.")
    item_id = TARGET_ITEM_ID or saved_id
    if item_id:
        item = gis.content.get(item_id)
        if item is None:
            raise RuntimeError("Saved target is missing/inaccessible. Restore it or choose a new SERVICE_NAME.")
    else:
        found = gis.content.search(query=f'owner:"{owner}" AND tags:"{service_tag}"',
                                   item_type="Feature Service", max_items=100)
        found = [i for i in found if i.owner == owner and service_tag in i.tags]
        if len(found) > 1:
            raise RuntimeError("Multiple matching targets; specify TARGET_ITEM_ID explicitly.")
        item = found[0] if found else None
        if item is None:
            item = gis.content.create_service(
                name=SERVICE_NAME, service_type="featureService", wkid=4326,
                has_static_data=False, max_record_count=2000,
                capabilities="Query,Create,Update,Delete,Editing",
                supported_query_formats="JSON,geoJSON",
                item_properties={"title": SERVICE_TITLE,
                    "tags": [service_tag, config_tag, "VIIRS", "WorldCover"],
                    "snippet": "Incrementally filtered VIIRS detections using the WorldCover 3×3 footprint filter.",
                    "description": "Rolling VIIRS vegetation hotspots; not confirmed fire events. "
                                   "Source: Esri/NASA VIIRS; ESA WorldCover 2021 v200. Countries supplied by owner.",
                    "access": "private"})
            if item is None:
                raise RuntimeError("Hosted service creation returned no item.")
            log(f"Created target item: {item.id}")
    if item.owner != owner or item.type != "Feature Service":
        raise RuntimeError("Run as the owner of this notebook's hosted Feature Service.")
    if service_tag not in item.tags or config_tag not in item.tags:
        raise RuntimeError("Target does not match this workflow/filter configuration. "
                           "For changed filters or an older notebook layer, use a new SERVICE_NAME and blank TARGET_ITEM_ID.")
    meta_set(db, "target_item_id", item.id)
    db.commit()  # Persist immediately, including recovery from incomplete initial setup.
    collection = FeatureLayerCollection.fromitem(item)
    if not collection.layers:
        require_success(collection.manager.add_to_definition({"layers": [layer_definition()]}), "Create output layer")
    layer = FeatureLayer(item.url.rstrip("/") + "/0", gis)
    props = layer.properties
    if props.geometryType != "esriGeometryPoint" or props.get("isView", False):
        raise RuntimeError("Target must be the notebook's source point layer, not a view.")
    if collection.properties.get("syncEnabled", False):
        raise RuntimeError("Disable sync on the output service; append upsert requires sync disabled.")
    actual = {f["name"]: f for f in props.fields}
    for expected in FIELDS:
        field = actual.get(expected["name"], {})
        if field.get("type") != expected["type"] or field.get("length", 0) < expected.get("length", 0):
            raise RuntimeError(f"Unexpected output schema for {expected['name']}; choose a new service name.")
    indexes = props.get("indexes", [])
    if not any(i.get("fields", "").lower() == "detection_id" and i.get("isUnique") for i in indexes):
        require_success(layer.manager.add_to_definition({"indexes": layer_definition()["indexes"][:1]}), "Add unique detection index")
        layer = FeatureLayer(layer.url, gis)
        if not any(i.get("fields", "").lower() == "detection_id" and i.get("isUnique")
                   for i in layer.properties.get("indexes", [])):
            raise RuntimeError("Service did not create the required unique detection_id index.")
    if not layer.properties.get("supportsAppend", False):
        raise RuntimeError("The output service does not support append.")
    if "featurecollection" not in str(layer.properties.get("supportedAppendFormats", "")).lower():
        raise RuntimeError("The output service cannot append featureCollection data.")
    if UPDATE_HOURS_OLD and not layer.properties.get("supportsCalculate", False):
        raise RuntimeError("Output does not support server-side hours_old calculation.")
    return item, layer


def hydrate_state(db, layer):
    # Needed once, or after lost workspace state. Only two attributes; no geometry.
    if meta_get(db, "hydrated", False):
        return
    records = layer.query(where="1=1", out_fields="detection_id,acq_time",
                          return_geometry=False, return_all_records=True).features
    db.executemany("INSERT OR REPLACE INTO seen VALUES (?, ?, 1)",
                   [(f.attributes["detection_id"], int(f.attributes["acq_time"])) for f in records])
    meta_set(db, "hydrated", True)
    db.commit()
    log(f"Recovered {len(records):,} accepted detection keys from output.")


def append_rows(layer, rows, now_ms):
    for batch in chunks(rows, APPEND_BATCH_SIZE):
        features = []
        for row in batch:
            if len(row["detection_id"]) > 160:
                raise ValueError("Source detection key exceeds 160 characters.")
            features.append({"geometry": {"x": row["longitude"], "y": row["latitude"],
                                          "spatialReference": {"wkid": 4326}},
                             "attributes": {"detection_id": row["detection_id"],
                                            "acq_time": row["acq_time"],
                                            "hours_old": round((now_ms-row["acq_time"])/HOUR_MS, 1) if UPDATE_HOURS_OLD else None,
                                            "frp": row["frp"], "country": row["country"],
                                            "landcover_center_class": row["landcover_center_class"]}})
        payload = {"layers": [{"layerDefinition": {"id": 0, "name": "viirs_batch",
                   "type": "Feature Layer", "geometryType": "esriGeometryPoint", "fields": DATA_FIELDS},
                   "featureSet": {"geometryType": "esriGeometryPoint", "spatialReference": {"wkid": 4326},
                                  "fields": DATA_FIELDS, "features": features}}]}
        # A unique index + upsert makes retries safe after a timeout or partial run.
        # Existing observations remain unchanged, matching the R workflow.
        result = layer.append(edits=payload, upload_format="featureCollection", upsert=True,
                              upsert_matching_field="detection_id", skip_updates=True,
                              update_geometry=False, rollback=True, return_messages=True)
        success = result[0] if isinstance(result, tuple) else result
        require_success(success, f"Append {len(batch)} detections: {result}")


def maintain_window(layer, now_ms):
    result = layer.delete_features(where=f"acq_time < {sql_time(now_ms - ROLLING_HOURS * HOUR_MS)}",
                                   return_delete_results=False)
    require_success(result, "Delete expired detections")
    if UPDATE_HOURS_OLD:
        result = layer.calculate(where="1=1", calc_expression={
            "field": "hours_old", "sqlExpression": f"ROUND(({sql_time(now_ms)} - acq_time) * 24, 1)"},
            sql_format="standard")
        require_success(result, "Recalculate hours_old")


## 7. Run

This cell defines orchestration; the next cell runs it. It validates your country
layer before creating the output, logs progress, and prints the output links and counts.
Even if no new points pass the filter, retention and age calculation still run.
An unavailable or empty source raises an error before retention is applied.

State and `last_success.json` are saved under the printed workspace directory.
A task failure is visible in ArcGIS Online's task results; it is not logged as success.

In [ ]:
def run_pipeline():
    from arcgis.gis import GIS
    validate_config()
    started = time.perf_counter()
    now_ms = int(time.time()) * 1000  # Single UTC reference; matches SQL second precision.
    gis = GIS("home")
    if gis.users.me is None:
        raise RuntimeError("Open this notebook in ArcGIS Online as its owner.")
    namespace = hashlib.sha256(
        f"{gis.properties.id}|{gis.users.me.username}|{SERVICE_NAME}".encode()).hexdigest()[:16]
    directory = Path(WORKSPACE_ROOT) / namespace
    with state_db(directory) as db:
        countries = country_layer(gis)  # Validate placeholder, layer ID, and field before creating output.
        item, layer = get_target(gis, db)
        hydrate_state(db, layer)
        rows, reconciled, source_info = fetch_source(db, now_ms)
        candidates = unknown_rows(db, rows)
        log(f"Downloaded {len(rows):,}; unseen {len(candidates):,}; cached {len(rows)-len(candidates):,}.")
        index, retained = None, 0
        for batch in chunks(candidates, PROCESS_BATCH_SIZE):
            accepted = vegetation_filter(batch)
            if accepted:
                if index is None:
                    index = country_index(gis, countries, directory, now_ms)
                assign_countries(accepted, index)
                append_rows(layer, accepted, now_ms)
            # Commit only after the entire batch's hosted writes succeeded.
            # Rejections are cached too; raster failures never become cached rejections.
            remember(db, batch, {r["detection_id"] for r in accepted})
            retained += len(accepted)
        maintain_window(layer, now_ms)
        total = layer.query(where="1=1", return_count_only=True)
        previous_success = meta_get(db, "last_success_ms")
        summary = {"completed_utc": datetime.now(timezone.utc).isoformat(),
                   "reference_utc": datetime.fromtimestamp(now_ms/1000, timezone.utc).isoformat(),
                   "downloaded": len(rows), "unseen": len(candidates), "new_retained": retained,
                   "features_in_layer": total, "rolling_hours": ROLLING_HOURS,
                   "full_reconciliation": reconciled, "source": source_info,
                   "seconds": round(time.perf_counter()-started, 2), "target_item_id": item.id,
                   "layer_url": layer.url}
        atomic_json(directory / "last_success.json", summary)
        with db:
            db.execute("DELETE FROM seen WHERE acq_time < ?", (now_ms - ROLLING_HOURS * HOUR_MS,))
            meta_set(db, "last_success_ms", now_ms)
            meta_set(db, "rolling_hours", ROLLING_HOURS)
            if reconciled or previous_success is None:
                meta_set(db, "last_reconcile_ms", now_ms)
        log(f"Complete: {retained:,} new accepted; {total:,} in hosted layer; {summary['seconds']} s.")
        print(f"Set TARGET_ITEM_ID = {item.id!r} in settings, then save the notebook.")
        print(f"Layer: https://www.arcgis.com/home/item.html?id={item.id}")
        print(f"Map Viewer: https://www.arcgis.com/apps/mapviewer/index.html?layers={item.id}")
        print(f"State and last successful run: {directory}")
        return summary


In [ ]:
summary = run_pipeline()
summary

## 8. Schedule every hour

1. Run the notebook manually, confirm the result, paste its printed `TARGET_ITEM_ID`
   into settings, and **save**.
2. Open **Tasks → Create task**. Set a recurring schedule for every **1 hour**.
3. In the advanced task settings, enable **Use workspace**. This is required to retain
   the SQLite state and country cache under `/arcgis/home` between scheduled containers.
4. Set the maximum run time to **55 minutes** and optionally enable **Update notebook
   on completion** to keep its latest output. Save the task. Check its first scheduled result.

The initial catch-up can take longer than a normal hourly run; complete it manually
before enabling the task. Keep only one task for this output. Monitor the reported
run duration before increasing worker counts or bootstrap size. Each run ends normally;
the ArcGIS task scheduler supplies the hourly cadence.

Use the printed hosted layer in Map Viewer or Dashboards. Set a suitable layer refresh
interval (for example, 5–15 minutes). Filter by the date field for rolling time windows.
`hours_old`, when enabled, represents age at the most recent successful update.
For a live age display in a compatible Arcade popup, use
`Round(DateDiff(Now(), $feature.acq_time, "hours"), 1)`.

## Operational notes

- **Filter changes:** use a new service name and blank target ID for a new variant.
  The notebook checks a configuration fingerprint to avoid mixing classifications.
- **Missed runs:** the next query expands automatically and the daily sweep catches
  older late arrivals, limited to source retention. This is a rolling dataset, not an archive.
- **Performance:** steady runs skip previously classified detections, cache country
  geometry, sample native raster blocks in parallel, and append only new detections.
  Full reconciliation and first-time bootstrap have higher I/O costs. Actual speed
  depends on hotspot volume, raster distribution, and the ArcGIS/ESA endpoints.
- **Hosting costs:** the output uses hosted-feature storage and notebook execution
  under your organization's credit rules. Standard runtime avoids an ArcPy dependency.
- **Interpretation:** thermal anomalies are possible vegetation fires, not confirmed
  fire incidents. WorldCover describes 2021 land cover. Changes since then and the
  approximate footprint can affect classification.

## References and validation scope

- [Production R workflow at the inspected commit](https://github.com/kengelmayer/VIIRS_Filter/tree/2993200916a64dcb0cd456d1fae5ad78bd4b7c97)
- [ArcGIS notebook scheduling and workspace persistence](https://doc.arcgis.com/en/arcgis-online/create-maps/prepare-a-notebook-for-automated-execution.htm)
- [ArcGIS Standard and Advanced runtimes](https://doc.arcgis.com/en/arcgis-online/create-maps/specify-the-runtime-of-a-notebook.htm)
- [FeatureLayer append/upsert API](https://developers.arcgis.com/python/latest/api-reference/arcgis.features.toc.html#arcgis.features.FeatureLayer.append)
- [ArcGIS SQL date calculations](https://doc.arcgis.com/en/arcgis-online/manage-data/calculate-fields.htm)
- [Rasterio block-aligned reads](https://rasterio.readthedocs.io/en/stable/topics/windowed-rw.html)
- [Shapely spatial indexing](https://shapely.readthedocs.io/en/stable/strtree.html)

The notebook's filtering, raster sampling, cache, retry behavior and payload construction
were tested locally. Public VIIRS and WorldCover reads were also checked. Creating,
appending to and scheduling a layer in **your** ArcGIS Online organization still requires
the first manual run after filling the placeholder; no authenticated deployment was
performed during notebook preparation.